In [ ]:
import cv2

print("Scanning for active camera channels (checking slots 0 to 8)...")
print("This might take a moment...")

active_channels = []

# Scan through potential USB indexes
for index in range(9):
    # Try opening the video stream channel
    cap = cv2.VideoCapture(index)
    
    # Check if the camera opened and can return a frame size
    if cap.isOpened():
        width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
        height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
        
        # Test if it actually delivers video pixels or if it is empty metadata
        ret, frame = cap.read()
        if ret:
            print(f"-> SUCCESS: Found active camera at Index [{index}] (Resolution: {int(width)}x{int(height)})")
            active_channels.append(index)
        else:
            print(f"   [Index {index}] Opened, but failed to stream image data.")
            
        cap.release()
    else:
        pass

print("\n--- SCAN COMPLETE ---")
if len(active_channels) >= 2:
    print(f"You have enough cameras! Try updating your script using indexes: {active_channels[:2]}")
elif len(active_channels) == 1:
    print(f"Only found one active camera stream at Index {active_channels}. Is the second camera unplugged?")
else:
    print("No cameras detected at all. Check your USB connection cables to Reachy's head.")


In [ ]:
import cv2
import numpy as np
import json

# 1. LOAD CALIBRATION PARAMETERS FROM YOUR JSON FILE
with open('../data/calibration/stereo_params_template.json', 'r') as f:
    config = json.load(f)

# Force raw distortion arrays into flat arrays with 5 elements
dist_l = np.array(config['camera_left']['distortion'], dtype=np.float64).flatten()
dist_r = np.array(config['camera_right']['distortion'], dtype=np.float64).flatten()

# Force Extrinsics into exact structural shapes
R = np.array(config['R'], dtype=np.float64).reshape(3, 3)
T = np.array(config['T'], dtype=np.float64).reshape(3, 1)

# Function to safely build a perfect 3x3 camera matrix
def make_camera_matrix(c_data):
    return np.array([
        [float(c_data['fx']), 0.0,                  float(c_data['cx'])],
        [0.0,                  float(c_data['fy']), float(c_data['cy'])],
        [0.0,                  0.0,                  1.0]
    ], dtype=np.float64).reshape(3, 3)

# Build the matrices and guarantee they are 3x3
mtx_l = make_camera_matrix(config['camera_left'])
mtx_r = make_camera_matrix(config['camera_right'])

# 2. COMPUTE RECTIFICATION MAPS (HORIZON ALIGNMENT)
img_shape = (int(config['image_width']), int(config['image_height']))

# The reshaped matrices will now cleanly pass through stereoRectify
R1, R2, P1, P2, Q, _, _ = cv2.stereoRectify(
    mtx_l, dist_l, mtx_r, dist_r, img_shape, R, T
)

# Create lookup maps for pixel remapping
map_l1, map_l2 = cv2.initUndistortRectifyMap(mtx_l, dist_l, R1, P1, img_shape, cv2.CV_16SC2)
map_r1, map_r2 = cv2.initUndistortRectifyMap(mtx_r, dist_r, R2, P2, img_shape, cv2.CV_16SC2)

# 3. CONFIGURE THE STEREO DEPTH MATCHING OBJECT
stereo = cv2.StereoSGBM_create(
    minDisparity=0,
    numDisparities=64,    # Must be divisible by 16
    blockSize=11,         # Search window size
    P1=8 * 3 * 11**2,
    P2=32 * 3 * 11**2,
    disp12MaxDiff=1,
    uniquenessRatio=15,
    speckleWindowSize=100,
    speckleRange=2
)

# 4. START LIVE CAMERA STREAM LOOP
cap_left = cv2.VideoCapture(4,cv2.CAP_V4L2)
cap_right = cv2.VideoCapture(0,cv2.CAP_V4L2)



# Set video sizes
#for cap in [cap_left, cap_right]:
#    cap.set(cv2.CAP_PROP_FRAME_WIDTH, img_shape[0])
#    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, img_shape[1])

# Use standard MJPEG compression to allow high resolution streams over USB safely
for cap in [cap_left, cap_right]:
    #cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'MJPG'))
    cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc('M', 'J', 'P', 'G'))
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

print("Streaming live depth map... Press 'q' inside the video window to exit.")

while True:
    ret_l, frame_l = cap_left.read()
    ret_r, frame_r = cap_right.read()
    
    if not ret_l or not ret_r:
        print("Error: Could not read video frames from cameras. Check camera indexes (0, 1, 2)!")
        break

    gray_l = cv2.cvtColor(frame_l, cv2.COLOR_BGR2GRAY)
    gray_r = cv2.cvtColor(frame_r, cv2.COLOR_BGR2GRAY)

    # Apply horizontal alignment mapping
    rectified_l = cv2.remap(gray_l, map_l1, map_l2, cv2.INTER_LINEAR)
    rectified_r = cv2.remap(gray_r, map_r1, map_r2, cv2.INTER_LINEAR)

    # Calculate disparity (depth)
    disparity = stereo.compute(rectified_l, rectified_r)

    # Normalize map for display
    depth_map = cv2.normalize(disparity, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    depth_colored = cv2.applyColorMap(depth_map, cv2.COLORMAP_JET)

    # Show live windows
    cv2.imshow("Left Video Feed (Rectified)", rectified_l)
    cv2.imshow("Live Depth Map Visualization", depth_colored)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap_left.release()
cap_right.release()
cv2.destroyAllWindows()


In [1]:
#TEST VIDEO FEED WORKS

import cv2

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cap.isOpened():
    print("Cannot open camera")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    # ROTATE 90 DEGREES CLOCKWISE TO FIX THE SIDEWAYS MOUNT
    # If it is upside down, change to cv2.ROTATE_90_COUNTERCLOCKWISE
    rotated_frame = cv2.rotate(frame, cv2.ROTATE_90_COUNTERCLOCKWISE)
    
    cv2.imshow('Reachy Eye Test', rotated_frame)
    
    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
